In [1]:
import os
import random
import pandas as pd
import numpy as np

In [2]:
SPLIT_DIR = "../datasets/splits"
OUTPUT_DIR = "../datasets/triplets"

os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_CSV = os.path.join(SPLIT_DIR, "train_images.csv")
VAL_CSV   = os.path.join(SPLIT_DIR, "val_images.csv")
TEST_CSV  = os.path.join(SPLIT_DIR, "test_images.csv")

SEED = 42

TRAIN_TRIPLETS_PER_ANCHOR = 8
VAL_TRIPLETS_PER_ANCHOR = 4
TEST_TRIPLETS_PER_ANCHOR = 4

random.seed(SEED)
np.random.seed(SEED)

In [3]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train images:", len(train_df))
print("Val images  :", len(val_df))
print("Test images :", len(test_df))

Train images: 1848
Val images  : 397
Test images : 397


In [4]:
print(train_df.head())

                                          image_path  \
0  ../datasets/sampled_sneakers_1024x1024\resized...   
1  ../datasets/sampled_sneakers_1024x1024\normali...   
2  ../datasets/sampled_sneakers_1024x1024\resized...   
3  ../datasets/sampled_sneakers_1024x1024\resized...   
4  ../datasets/sampled_sneakers_1024x1024\resized...   

                           image_name        brand  
0         resized_padded_vans_167.jpg         vans  
1              normalized_nike_15.jpg         nike  
2       resized_padded_reebok_415.jpg       reebok  
3            resized_all_vans_575.jpg         vans  
4  resized_padded_underarmour_269.jpg  underarmour  


In [5]:
print(train_df.columns.tolist())

['image_path', 'image_name', 'brand']


In [6]:
print("TRAIN")
print(train_df["brand"].value_counts())
print()

print("VALIDATION")
print(val_df["brand"].value_counts())
print()

print("TEST")
print(test_df["brand"].value_counts())

TRAIN
brand
vans           643
puma           383
nike           265
reebok         239
underarmour    160
newbalance     158
Name: count, dtype: int64

VALIDATION
brand
vans           138
puma            82
nike            57
reebok          51
underarmour     35
newbalance      34
Name: count, dtype: int64

TEST
brand
vans           138
puma            82
nike            57
reebok          51
underarmour     35
newbalance      34
Name: count, dtype: int64


In [7]:
def group_by_brand(df):
    return {
        brand: group["image_name"].tolist()
        for brand, group in df.groupby("brand")
    }

In [8]:
train_groups = group_by_brand(train_df)
val_groups = group_by_brand(val_df)
test_groups = group_by_brand(test_df)

In [9]:
for brand, imgs in train_groups.items():
    print(brand, len(imgs))

newbalance 158
nike 265
puma 383
reebok 239
underarmour 160
vans 643


In [10]:
def generate_triplets(df, trips_per_anchor, seed=42):
    
    rng = random.Random(seed)
    
    groups = group_by_brand(df)
    all_brands = list(groups.keys())
    
    triplets = []
    
    for _, row in df.iterrows():
        
        anchor = row["image_name"]
        anchor_brand = row["brand"]
        
        # All possible positive images from same brand
        positives = [
            img for img in groups[anchor_brand]
            if img != anchor
        ]
        
        # Brands different from anchor brand
        negative_brands = [
            b for b in all_brands
            if b != anchor_brand
        ]
        
        if len(positives) == 0:
            continue
        
        for _ in range(trips_per_anchor):
            
            # Random same-brand positive
            positive = rng.choice(positives)
            
            # Random different-brand
            negative_brand = rng.choice(negative_brands)
            
            # Random image from that brand
            negative = rng.choice(groups[negative_brand])
            
            triplets.append({
                "anchor_image": anchor,
                "positive_image": positive,
                "negative_image": negative,
                "anchor_brand": anchor_brand,
                "negative_brand": negative_brand
            })
    
    return pd.DataFrame(triplets)

In [11]:
train_triplets = generate_triplets(
    train_df,
    trips_per_anchor=TRAIN_TRIPLETS_PER_ANCHOR,
    seed=SEED
)

print("Training triplets:", len(train_triplets))

Training triplets: 14784


In [12]:
val_triplets = generate_triplets(
    val_df,
    trips_per_anchor=VAL_TRIPLETS_PER_ANCHOR,
    seed=SEED + 1
)

print("Validation triplets:", len(val_triplets))

Validation triplets: 1588


In [19]:
test_triplets = generate_triplets(
    test_df,
    trips_per_anchor=TEST_TRIPLETS_PER_ANCHOR,
    seed=SEED + 1
)

print("Test triplets:", len(test_triplets))

Test triplets: 1588


In [13]:
train_triplets.head(10)

,anchor_image,positive_image,negative_image,anchor_brand,negative_brand
0,resized_padded_vans_167.jpg,resized_all_vans_332.jpg,resized_padded_new_newbalance_123.jpg,vans,newbalance
1,resized_padded_vans_167.jpg,resized_padded_vans_113.jpg,normalized_nike_201.jpg,vans,nike
2,resized_padded_vans_167.jpg,resized_all_vans_27.jpg,resized_padded_underarmour_235.jpg,vans,underarmour
3,resized_padded_vans_167.jpg,resized_all_vans_431.jpg,resized_padded_reebok_365.jpg,vans,reebok
4,resized_padded_vans_167.jpg,resized_all_vans_280.jpg,resized_padded_new_newbalance_247.jpg,vans,newbalance
5,resized_padded_vans_167.jpg,resized_padded_vans_446.jpg,resized_padded_underarmour_123.jpg,vans,underarmour
6,resized_padded_vans_167.jpg,resized_all_vans_704.jpg,resized_padded_underarmour_208.jpg,vans,underarmour
7,resized_padded_vans_167.jpg,resized_all_vans_266.jpg,resized_padded_reebok_290.jpg,vans,reebok
8,normalized_nike_15.jpg,normalized_nike_353.jpg,resized_padded_vans_347.jpg,nike,vans
9,normalized_nike_15.jpg,normalized_nike_222.jpg,resized_padded_puma_46.jpg,nike,puma


In [14]:
print(
    "Invalid positive pairs:",
    (train_triplets["anchor_brand"] != train_triplets["positive_image"]
        .str.lower()
        .str.extract(
            "(nike|puma|vans|newbalance|underarmour|reebok)",
            expand=False
        )
    ).sum()
)

Invalid positive pairs: 0


In [16]:
def create_brand_map(df):
    return dict(zip(df["image_name"], df["brand"]))

train_brand_map = create_brand_map(train_df)
val_brand_map = create_brand_map(val_df)
test_brand_map = create_brand_map(test_df)

In [17]:
def validate_triplets(triplets, brand_map):
    
    errors = []
    
    for _, row in triplets.iterrows():
        
        a = row["anchor_image"]
        p = row["positive_image"]
        n = row["negative_image"]
        
        if a not in brand_map:
            errors.append(("missing_anchor", a))
            continue
            
        if p not in brand_map:
            errors.append(("missing_positive", p))
            continue
            
        if n not in brand_map:
            errors.append(("missing_negative", n))
            continue
        
        # Anchor and positive MUST have same brand
        if brand_map[a] != brand_map[p]:
            errors.append(("wrong_positive", a, p))
        
        # Anchor and negative MUST have different brands
        if brand_map[a] == brand_map[n]:
            errors.append(("wrong_negative", a, n))
    
    return errors

In [20]:
train_errors = validate_triplets(train_triplets, train_brand_map)
val_errors = validate_triplets(val_triplets, val_brand_map)
test_errors = validate_triplets(test_triplets, test_brand_map)

print("Train errors:", len(train_errors))
print("Val errors  :", len(val_errors))
print("Test errors :", len(test_errors))

Train errors: 0
Val errors  : 0
Test errors : 0


In [21]:
train_images = set(train_df["image_name"])
val_images = set(val_df["image_name"])
test_images = set(test_df["image_name"])

print("Train ∩ Val :", len(train_images & val_images))
print("Train ∩ Test:", len(train_images & test_images))
print("Val ∩ Test  :", len(val_images & test_images))

Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0


In [22]:
def check_triplet_split(triplets, allowed_images):
    
    used_images = set(
        triplets["anchor_image"]
    ) | set(
        triplets["positive_image"]
    ) | set(
        triplets["negative_image"]
    )
    
    outside = used_images - allowed_images
    
    return outside

In [23]:
train_outside = check_triplet_split(
    train_triplets,
    train_images
)

val_outside = check_triplet_split(
    val_triplets,
    val_images
)

test_outside = check_triplet_split(
    test_triplets,
    test_images
)

print("Train outside split:", len(train_outside))
print("Val outside split  :", len(val_outside))
print("Test outside split :", len(test_outside))

Train outside split: 0
Val outside split  : 0
Test outside split : 0


In [24]:
print(train_triplets["anchor_brand"].value_counts())
print(train_triplets["negative_brand"].value_counts())

anchor_brand
vans           5144
puma           3064
nike           2120
reebok         1912
underarmour    1280
newbalance     1264
Name: count, dtype: int64
negative_brand
newbalance     2760
underarmour    2651
reebok         2549
nike           2540
puma           2304
vans           1980
Name: count, dtype: int64


In [25]:
train_triplets.to_csv(
    os.path.join(OUTPUT_DIR, "train_triplets.csv"),
    index=False
)

val_triplets.to_csv(
    os.path.join(OUTPUT_DIR, "val_triplets.csv"),
    index=False
)

test_triplets.to_csv(
    os.path.join(OUTPUT_DIR, "test_triplets.csv"),
    index=False
)

print("Triplet datasets saved!")

Triplet datasets saved!
